# Intelli-Speech — TRAIN_V1 (ZF-Net + GMM, version 1)

Runs the v1 experiments on the Colab runtime: baselines → GMM noise classifier → ZF-Net mask
estimator → test-split evaluation → results table → listening check.

**Run order:** cells 1 → 12 from top to bottom. If cell 3 asks for a restart, restart the
kernel and continue from cell 4. Every cell skips work whose outputs already exist, so re-running
is safe.

**Writes only** under `Intelli-Speech-data/experiments/`. The dataset `paired_dataset_v3` is
read-only; the scripts refuse to write inside it. All research logic lives in `intelli_speech/`
and `scripts/`; this notebook only orchestrates them.

In [1]:
# ============================================================
# CELL 1 | RUNTIME
# ============================================================
import platform, sys
import numpy as np
print("Python:", platform.python_version())
print("NumPy :", np.__version__)
try:
    import torch
    GPU = torch.cuda.is_available()
    print("torch :", torch.__version__, "| CUDA:", GPU, "|", torch.cuda.get_device_name(0) if GPU else "CPU only")
except Exception as exc:
    GPU = False
    print("torch unavailable:", exc)

EPOCHS = 20 if GPU else 8
CROP_SEC = 2.0 if GPU else 1.5
BATCH = 16 if GPU else 8
if not GPU:
    print("WARNING: no GPU. ZF-Net training will be slow (several minutes per epoch). "
          "Runtime > Change runtime type > T4 GPU is recommended.")
print(f"Training plan: epochs={EPOCHS} crop={CROP_SEC}s batch={BATCH}")

Python: 3.13.16
NumPy : 2.1.3
torch : 2.11.0+cu130 | CUDA: True | Tesla T4
Training plan: epochs=20 crop=2.0s batch=16


In [2]:
# ============================================================
# CELL 2 | DRIVE MOUNT + REPOSITORY SYNC (explicit pull)
# ============================================================
import os, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
REPO_DIR = Path("/content/Intelli-Speech")
REPO_URL = "https://github.com/niharika9nx/Intelli-Speech.git"

if not (DRIVE_ROOT / "MyDrive").is_dir():
    from google.colab import drive
    drive.mount(str(DRIVE_ROOT))

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"],
                           capture_output=True, text=True).stdout.strip()
    if dirty:
        print("Local changes in the Colab checkout (not pulling over them):\n" + dirty)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())
assert (REPO_DIR / "scripts" / "train_zfnet.py").is_file(), "Checkout is missing the v1 code; pull failed?"
print("v1 code present.")

Mounted at /content/drive
102ace8 Add v1 ZF-Net + GMM model, training/eval scripts, inference server, frontend wiring
v1 code present.


In [3]:
# ============================================================
# CELL 3 | INSTALL BASELINE + MODEL ENVIRONMENT
# ============================================================
import importlib.metadata as md
numpy_before = md.version("numpy")
!pip install -q -r requirements-colab.txt
numpy_after = md.version("numpy")
!python scripts/setup_colab_runtime.py
print(f"NumPy before={numpy_before} after={numpy_after}")
if numpy_before != numpy_after:
    print("\n>>> NumPy changed. Restart the kernel now, then continue from CELL 4. <<<")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INTELLI-SPEECH COLAB RUNTIME DIAGNOSTICS
Python: 3.13.16 (/usr/bin/python3)
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
NumPy: 2.1.3
Numba: 0.61.2
librosa: 0.10.2.post1
PyTorch: 2.11.0+cu130
torchaudio: 2.11.0+cu130
ClearerVoice: not installed
Compatibility warning: ClearerVoice 0.1.2 declares NumPy <2.0; package compatibility is not yet established.
CUDA available: True
clearvoice:
  installed=not installed
  importable=False
librosa:
  installed=0.10.2.post1
  importable=True
  relevant declared requirements:
    numpy !=1.22.0,!=1.22.1,!=1.22.2,>=1.20.3
    soundfile >=0.12.1
    numpydoc ; extra == 'docs'
numpy:
  installed=2.1.3
  importable=True
Google Colab environment: True
Drive mount failed: 'NoneType' object has no attribute 'kernel'
NumPy before=2.1.3 after=2.1.3


In [4]:
# ============================================================
# CELL 4 | CONFIG + DATASET CHECK (read-only)
# ============================================================
import os, sys, json
from pathlib import Path
REPO_DIR = Path("/content/Intelli-Speech"); os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
try:
    GPU, EPOCHS, CROP_SEC, BATCH
except NameError:  # kernel was restarted after cell 3
    import torch
    GPU = torch.cuda.is_available()
    EPOCHS, CROP_SEC, BATCH = (20, 2.0, 16) if GPU else (8, 1.5, 8)

import librosa, numpy as np
from intelli_speech.dataset import load_metadata, split_counts

D = Path("/content/drive/MyDrive/Intelli-Speech-data/paired_dataset_v3")
EXP = D.parent / "experiments"
R = EXP / "zfnet_gmm_v1"
print("NumPy", np.__version__, "| librosa", librosa.__version__, "| GPU", GPU)

meta = load_metadata(D)
print("rows:", len(meta), "| splits:", split_counts(meta))
assert "noise_environment" in meta.columns, "metadata.csv has no noise_environment column"
print("noise classes:", meta["noise_environment"].value_counts().to_dict())

NumPy 2.1.3 | librosa 0.10.2.post1 | GPU True
rows: 1000 | splits: {'train': 800, 'validation': 100, 'test': 100}
noise classes: {'DKITCHEN': 200, 'NPARK': 200, 'OOFFICE': 200, 'STRAFFIC': 200, 'TBUS': 200}


In [5]:
# ============================================================
# CELL 5 | BASELINES ON THE TEST SPLIT (Wiener + oracle IRM)
# ============================================================
for method in ["wiener", "oracle-irm"]:
    out = EXP / f"{method}_test"
    if (out / "summary.json").is_file():
        print(f"[skip] {method}: {out}/summary.json exists")
        continue
    !python scripts/evaluate_baselines.py --dataset-dir "{D}" --split test --method {method} --include-pesq --include-stoi

[1/100] tef_08664_01599099513 target=5 dB input=5.000 dB output=9.746 dB change=+4.746 dB
[2/100] tef_08664_01599099513 target=10 dB input=10.000 dB output=15.524 dB change=+5.524 dB
[3/100] tef_04213_01725774589 target=5 dB input=5.000 dB output=7.417 dB change=+2.417 dB
[4/100] tef_04213_01725774589 target=10 dB input=10.000 dB output=14.496 dB change=+4.496 dB
[5/100] tef_05705_00933490011 target=5 dB input=5.000 dB output=5.869 dB change=+0.869 dB
[6/100] tef_05705_00933490011 target=10 dB input=10.000 dB output=9.722 dB change=-0.278 dB
[7/100] tef_05484_00724275999 target=5 dB input=5.000 dB output=9.888 dB change=+4.888 dB
[8/100] tef_05484_00724275999 target=10 dB input=10.000 dB output=13.968 dB change=+3.968 dB
[9/100] tef_04213_01598208852 target=5 dB input=5.000 dB output=7.084 dB change=+2.084 dB
[10/100] tef_04213_01598208852 target=10 dB input=10.000 dB output=15.135 dB change=+5.135 dB
[11/100] tef_08664_00887585353 target=5 dB input=5.000 dB output=12.139 dB change=+7.

In [6]:
# ============================================================
# CELL 6 | TRAIN THE GMM NOISE CLASSIFIER
# ============================================================
if (R / "gmm.joblib").is_file():
    print("[skip] gmm.joblib exists")
else:
    !python scripts/train_gmm.py --dataset-dir "{D}" --output-dir "{R}"
report = json.loads((R / "gmm_report.json").read_text())
print("validation accuracy:", round(report["validation_accuracy"], 4), "| by SNR:", report["validation_accuracy_by_snr"])
print("temperature:", report["temperature"])
import pandas as pd
print(pd.DataFrame(report["validation_confusion"]).T.to_string())

[features] 100/800 train files
[features] 200/800 train files
[features] 300/800 train files
[features] 400/800 train files
[features] 500/800 train files
[features] 600/800 train files
[features] 700/800 train files
[features] 800/800 train files
[fit] 5 classes: ['DKITCHEN', 'NPARK', 'OOFFICE', 'STRAFFIC', 'TBUS']
GMM validation accuracy: 0.920 (temperature 2.328); saved /content/drive/.shortcut-targets-by-id/1W9xt6UZaltMCWYwwxy_cly8t0_nFeaNM/Intelli-Speech-data/experiments/zfnet_gmm_v1/gmm.joblib
validation accuracy: 0.92 | by SNR: {'5dB': 0.94, '10dB': 0.9}
temperature: 2.3278617175760243
          DKITCHEN  NPARK  OOFFICE  STRAFFIC  TBUS
DKITCHEN        24      0        0         0     0
NPARK            0     23        0         0     0
OOFFICE          0      2       11         0     3
STRAFFIC         0      1        0        15     0
TBUS             0      0        2         0    19


In [7]:
# ============================================================
# CELL 7 | ZF-NET SMOKE RUN (1 epoch, 5 steps, separate directory)
# ============================================================
SMOKE = EXP / "zfnet_gmm_v1_smoke"
!python scripts/train_zfnet.py --dataset-dir "{D}" --output-dir "{SMOKE}" --gmm "{R}/gmm.joblib" --epochs 1 --max-steps-per-epoch 5 --val-limit 10 --batch-size {BATCH} --overwrite

[load] 200/800
[load] 400/800
[load] 600/800
[load] 800/800
[load] 10/10
[model] zfnet-gmm-v1 params=190,673 device=cuda train=800 val=10
/content/Intelli-Speech/scripts/train_zfnet.py:216: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  train_losses.append(float(loss))
[epoch 1] train 0.1636 val 0.1618 val dSNR +2.53 dB (18.6s) *saved
{'model_version': 'zfnet-gmm-v1', 'best_epoch': 1, 'best_val_loss': 0.16181782484054566, 'best_val_snr_improvement_db': 2.525351333618164, 'epochs_run': 1, 'seconds': 18.7, 'checkpoint': '/content/drive/.shortcut-targets-by-id/1W9xt6UZaltMCWYwwxy_cly8t0_nFeaNM/Intelli-Speech-data/experiments/zfnet_gmm_v1_smoke/zfnet.pt'}


In [8]:
# ============================================================
# CELL 8 | ZF-NET FULL TRAINING
# ============================================================
if (R / "train_summary.json").is_file():
    print("[skip] training already finished:", (R / "train_summary.json").read_text())
else:
    !python scripts/train_zfnet.py --dataset-dir "{D}" --output-dir "{R}" --epochs {EPOCHS} --crop-sec {CROP_SEC} --batch-size {BATCH} --overwrite
import pandas as pd
print(pd.read_csv(R / "history.csv").to_string(index=False))

[load] 200/800
[load] 400/800
[load] 600/800
[load] 800/800
[load] 100/100
[model] zfnet-gmm-v1 params=190,673 device=cuda train=800 val=100
/content/Intelli-Speech/scripts/train_zfnet.py:216: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  train_losses.append(float(loss))
  epoch 1 step 25/50 loss 0.1334
  epoch 1 step 50/50 loss 0.1107
[epoch 1] train 0.1220 val 0.1125 val dSNR +7.32 dB (11.4s) *saved
  epoch 2 step 25/50 loss 0.0969
  epoch 2 step 50/50 loss 0.0814
[epoch 2] train 0.0891 val 0.1132 val dSNR +12.76 dB (10.7s)
  epoch 3 step 25/50 loss 0.0766
  epoch 3 step 50/50 loss 0.0691
[epoch 3] train 0.0728 val 0.0770 val dSNR +13.41 dB (10.8s) *saved
  epoch 4 step 25/50 loss 0.0632
  epoch 4 step 50/50 loss 0.0568
[epoch 4] train 0.0600 val 0.0844 val dSNR +13.49 dB (10.9s)
  epoch 5 ste

In [9]:
# ============================================================
# CELL 9 | EVALUATE ZF-NET + GMM ON THE TEST SPLIT
# ============================================================
out = EXP / "zfnet-gmm_test"
if (out / "summary.json").is_file():
    print("[skip] zfnet-gmm_test/summary.json exists")
else:
    !python scripts/evaluate_baselines.py --dataset-dir "{D}" --split test --method zfnet-gmm --gmm-checkpoint "{R}/gmm.joblib" --zfnet-checkpoint "{R}/zfnet.pt" --include-pesq --include-stoi

[1/100] tef_08664_01599099513 target=5 dB input=5.000 dB output=20.505 dB change=+15.505 dB
[2/100] tef_08664_01599099513 target=10 dB input=10.000 dB output=30.996 dB change=+20.996 dB
[3/100] tef_04213_01725774589 target=5 dB input=5.000 dB output=25.065 dB change=+20.065 dB
[4/100] tef_04213_01725774589 target=10 dB input=10.000 dB output=19.216 dB change=+9.216 dB
[5/100] tef_05705_00933490011 target=5 dB input=5.000 dB output=16.862 dB change=+11.862 dB
[6/100] tef_05705_00933490011 target=10 dB input=10.000 dB output=22.318 dB change=+12.318 dB
[7/100] tef_05484_00724275999 target=5 dB input=5.000 dB output=19.702 dB change=+14.702 dB
[8/100] tef_05484_00724275999 target=10 dB input=10.000 dB output=28.607 dB change=+18.607 dB
[9/100] tef_04213_01598208852 target=5 dB input=5.000 dB output=25.797 dB change=+20.797 dB
[10/100] tef_04213_01598208852 target=10 dB input=10.000 dB output=26.136 dB change=+16.136 dB
[11/100] tef_08664_00887585353 target=5 dB input=5.000 dB output=16.89

In [10]:
# ============================================================
# CELL 10 | RESULTS TABLE (test split)
# ============================================================
import pandas as pd
labels = {"wiener": "Wiener (non-oracle)", "zfnet-gmm": "ZF-Net + GMM (v1)",
          "oracle-irm": "Oracle IRM (upper bound, not deployable)"}
rows = []
for method, label in labels.items():
    path = EXP / f"{method}_test" / "metrics.csv"
    if not path.is_file():
        print("missing:", path); continue
    df = pd.read_csv(path)
    for snr, group in [("all", df)] + [(f"{s:g} dB", g) for s, g in df.groupby("target_snr_db")]:
        rows.append({"method": label, "input SNR": snr, "n": len(group),
                     "ΔSNR dB": group["snr_improvement_db"].mean(),
                     "PESQ": group["pesq"].mean() if group["pesq"].notna().any() else float("nan"),
                     "STOI": group["stoi"].mean() if group["stoi"].notna().any() else float("nan")})
table = pd.DataFrame(rows)
print(table.to_markdown(index=False, floatfmt=".3f"))
zsum = EXP / "zfnet-gmm_test" / "summary.json"
if zsum.is_file():
    acc = json.loads(zsum.read_text())["summary"].get("noise_classification_accuracy")
    print(f"\nGMM noise-type accuracy on test: {acc:.3f}")

| method                                   | input SNR   |   n |   ΔSNR dB |   PESQ |   STOI |
|:-----------------------------------------|:------------|----:|----------:|-------:|-------:|
| Wiener (non-oracle)                      | all         | 100 |     2.640 |  1.587 |  0.934 |
| Wiener (non-oracle)                      | 5 dB        |  50 |     3.705 |  1.480 |  0.915 |
| Wiener (non-oracle)                      | 10 dB       |  50 |     1.575 |  1.694 |  0.954 |
| ZF-Net + GMM (v1)                        | all         | 100 |    14.561 |  2.721 |  0.961 |
| ZF-Net + GMM (v1)                        | 5 dB        |  50 |    15.122 |  2.489 |  0.942 |
| ZF-Net + GMM (v1)                        | 10 dB       |  50 |    14.000 |  2.952 |  0.979 |
| Oracle IRM (upper bound, not deployable) | all         | 100 |    17.357 |  4.028 |  0.992 |
| Oracle IRM (upper bound, not deployable) | 5 dB        |  50 |    17.886 |  3.906 |  0.989 |
| Oracle IRM (upper bound, not deployable) | 10 dB

In [11]:
# ============================================================
# CELL 11 | LISTENING CHECK (3 test samples)
# ============================================================
from IPython.display import Audio, Markdown, display
from intelli_speech.audio_io import load_audio
from intelli_speech.baselines import wiener_enhance
from intelli_speech.dataset import load_pairs
from intelli_speech.enhancer import Enhancer
from intelli_speech.stft import STFTConfig

enhancer = Enhancer.from_checkpoints(R / "gmm.joblib", R / "zfnet.pt")
for pair in load_pairs(D, split="test")[:6:2]:
    clean, _ = load_audio(pair.clean_path)
    noisy, _ = load_audio(pair.noisy_path)
    out = enhancer.enhance(noisy)
    display(Markdown(f"**{pair.source_id}** | {pair.noise_environment} @ {pair.target_snr_db:g} dB "
                     f"| GMM says **{out.noise_type}** ({out.noise_probs[out.noise_type]:.0%})"))
    for name, audio in [("noisy", noisy), ("Wiener", wiener_enhance(noisy, STFTConfig()).enhanced),
                        ("ZF-Net + GMM", out.waveform), ("clean", clean)]:
        display(Markdown(name)); display(Audio(audio, rate=16000))

**tef_08664_01599099513** | NPARK @ 5 dB | GMM says **NPARK** (95%)

noisy

Wiener

ZF-Net + GMM

clean

**tef_04213_01725774589** | DKITCHEN @ 5 dB | GMM says **DKITCHEN** (100%)

... (5 outputs left out)

noisy

Wiener

ZF-Net + GMM

clean

## CELL 12 | Use the trained model in the app (local machine)

1. Copy `Intelli-Speech-data/experiments/zfnet_gmm_v1/gmm.joblib` and `zfnet.pt` from Google Drive
   into the repository's git-ignored `checkpoints/` folder on the local machine.
2. Start the inference server from the repository root:
   `INTELLI_SPEECH_GMM_PATH=checkpoints/gmm.joblib INTELLI_SPEECH_ZFNET_PATH=checkpoints/zfnet.pt python -m uvicorn server.app:app --port 8000`
3. Start the app: `cd app && VITE_ENHANCE_API_URL=http://127.0.0.1:8000 npm run dev`

Record the results table from CELL 10 in `PROJECT_NOTES.md`; do not create new Markdown files.